# Check the Firm Summaries

| Item | Description |
|---|---|
| Goal | Put the output of one T3.7.4 run (one input set, one summary family, one prompt version) into the firm-summary table and check it: one summary per firm of the set, none empty, all within the embedding limit, built from the right number of yearly summaries. |
| Input | `<scratch>/T3.7 - Firm Summaries/Interim/firm_summaries/<set>/<family>/<version>/firms.jsonl` and `firms.json` from T3.7.4; `output/T3.4.1/filings.csv`. |
| Output | `<scratch>/T3.7 - Firm Summaries/Processed/firm_summaries.parquet`: one row per firm, set, family, and prompt version; rows of other runs already in the file are kept, rows of this run are replaced. `output/T3.7.5/firm_summary_check_<set>_<family>_<version>.csv` and `example_firm_summaries_<set>_<family>_<version>.csv`. |
| Unit | A firm's summary under one family, built from its filings in the set. |
| Checks | Every firm of the set has exactly one summary; no summary is empty; every summary is within 512 bge tokens; `n_filings` equals the firm's filings in the set; every record carries the run's prompt version. Counted, not enforced: summaries trimmed to the limit by T3.7.4's last-resort rule, length by number of filings (a firm with one filing gets a rewrite of one yearly summary, which may pad), and whether the summary refers to its own inputs. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and the run to check

Change `SET`, `FAMILY`, and `VERSION` to check another run; the table on scratch accumulates runs.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.7.5"  # the identifier names this notebook's output folder
STAGE_FOLDER = SCRATCH_FOLDER / "T3.7 - Firm Summaries"
FIRM_FOLDER = STAGE_FOLDER / "Interim" / "firm_summaries"
TABLE_FILE = STAGE_FOLDER / "Processed" / "firm_summaries.parquet"
FILINGS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.4.1/filings.csv"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

SET = "all_years"
FAMILY = "summary_lines_of_business"
VERSION = "item1-mapreduce-1.3"  # the prompt version the run was made with
MAX_BGE_TOKENS = 512  # the encoder's input limit every summary must respect
N_EXAMPLES = 5  # summaries shown and saved as examples
RANDOM_SEED = 0
SAMPLE_FLAG = {"three_years": "in_three_years", "all_years": "in_all_years"}[SET]
run_folder = FIRM_FOLDER / SET / FAMILY / VERSION
TABLE_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Run: set {SET}, family {FAMILY}, prompt version {VERSION}")
print(f"Run folder: {run_folder}")
print(f"Outputs: {OUTPUT_FOLDER}")


Run: set all_years, family summary_lines_of_business, prompt version item1-mapreduce-1.3
Run folder: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Interim/firm_summaries/all_years/summary_lines_of_business/item1-mapreduce-1.3
Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.5


## Step 3. Load the firm summaries and the run record

One line per firm in `firms.jsonl`; a firm that was summarized in two sessions appears twice and the later record wins.


In [3]:
summaries = pd.DataFrame([json.loads(line) for line in (run_folder / "firms.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()])
record = json.loads((run_folder / "firms.json").read_text(encoding="utf-8"))
n_duplicates = int(summaries.duplicated("gvkey", keep="last").sum())
summaries = summaries.drop_duplicates("gvkey", keep="last").reset_index(drop=True)
print(f"Firm summaries read: {len(summaries):,}; duplicate records dropped: {n_duplicates}")
print(f"Run record: {record['firms_done_this_run']:,} firms this run, {record['firms_skipped_as_done']:,} skipped as done, {record['model_calls']:,} model calls on {record['gpu']}, {record['seconds'] / 60:.1f} minutes")
display(summaries.drop(columns=["summary"]).head(3))


Firm summaries read: 1,058; duplicate records dropped: 0
Run record: 1 firms this run, 1,057 skipped as done, 1 model calls on NVIDIA A40, 0.4 minutes


,gvkey,conm,set,family,prompt_version,n_filings,years,n_words,n_bge_tokens,within_bge_limit,reduce_rounds,run_started,trimmed
0,001013,ADC TELECOMMUNICATIONS INC,all_years,summary_lines_of_business,item1-mapreduce-1.3,12,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",319,446,True,1,2026-10-26T18:03:17+00:00,NaN
1,001045,AMERICAN AIRLINES GROUP INC,all_years,summary_lines_of_business,item1-mapreduce-1.3,22,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",251,322,True,1,2026-10-26T18:03:17+00:00,NaN
2,001075,PINNACLE WEST CAPITAL CORP,all_years,summary_lines_of_business,item1-mapreduce-1.3,15,"[1999, 2000, 2001, 2002, 2003, 2006, 2007, 201...",247,321,True,1,2026-10-26T18:03:17+00:00,NaN


## Step 4. Check coverage against the sample

Every firm with a filing in the set has exactly one summary, and `n_filings` equals the firm's filings in the set.


In [4]:
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
in_set = filings.loc[filings[SAMPLE_FLAG].eq(1)]
expected = in_set.groupby("gvkey").agg(n_filings_expected=("accessionNo", "size"), years_expected=("fyear", lambda s: sorted(int(y) for y in s)))
missing = sorted(set(expected.index) - set(summaries["gvkey"]))
extra = sorted(set(summaries["gvkey"]) - set(expected.index))
assert not missing, f"{len(missing)} firms of the set have no summary: {missing[:5]}"
assert not extra, f"{len(extra)} summaries belong to firms outside the set: {extra[:5]}"
assert summaries["gvkey"].is_unique
summaries = summaries.merge(expected, left_on="gvkey", right_index=True, how="left")
assert summaries["n_filings"].eq(summaries["n_filings_expected"]).all()
assert all(list(a) == b for a, b in zip(summaries["years"], summaries["years_expected"]))
summaries = summaries.drop(columns=["n_filings_expected", "years_expected"])
print(f"Firms in the set: {len(expected):,}; summaries: {len(summaries):,}; missing: {len(missing)}; extra: {len(extra)}")
display(summaries["n_filings"].value_counts().sort_index().rename("firms").to_frame().T)


Firms in the set: 1,058; summaries: 1,058; missing: 0; extra: 0


n_filings,1,2,3,4,5,6,7,8,9,10,...,18,19,20,21,22,23,24,25,26,27
firms,32,38,19,18,23,35,35,27,37,28,...,25,28,22,27,23,20,23,27,46,340


## Step 5. Check the summaries themselves

None empty, all within the embedding limit, all under the run's prompt version; the re-asks show how often the length rule bit.


In [5]:
assert summaries["summary"].str.strip().str.len().gt(0).all(), "an empty summary"
assert summaries["prompt_version"].eq(VERSION).all() and summaries["family"].eq(FAMILY).all() and summaries["set"].eq(SET).all()
assert summaries["n_bge_tokens"].le(MAX_BGE_TOKENS).all(), "a summary over the encoder limit"
assert summaries["within_bge_limit"].all()
print(f"Words per summary: min {summaries['n_words'].min()}, median {int(summaries['n_words'].median())}, max {summaries['n_words'].max()}")
print(f"bge tokens per summary: min {summaries['n_bge_tokens'].min()}, median {int(summaries['n_bge_tokens'].median())}, max {summaries['n_bge_tokens'].max()} (limit {MAX_BGE_TOKENS})")
summaries["trimmed"] = summaries["trimmed"].fillna(False).astype(bool) if "trimmed" in summaries else False  # runs before the trim rule have no field
print(f"Firms re-asked shorter: {int(summaries['reduce_rounds'].gt(1).sum()):,} of {len(summaries):,}; trimmed to the limit by dropping trailing sentences: {int(summaries['trimmed'].sum()):,}")


Words per summary: min 126, median 248, max 401
bge tokens per summary: min 173, median 328, max 511 (limit 512)
Firms re-asked shorter: 2 of 1,058; trimmed to the limit by dropping trailing sentences: 0


/scratch1/acroh/tmp/ipykernel_3110321/3480826883.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  summaries["trimmed"] = summaries["trimmed"].fillna(False).astype(bool) if "trimmed" in summaries else False  # runs before the trim rule have no field


## Step 6. Count length by number of filings and references to the inputs

A firm with one filing in the set gets a rewrite of a single yearly summary, so its length against the yearly summary shows whether the rewrite padded. The input-reference pattern is the one T3.7.3 counts on the filing summaries.


In [6]:
REFERS_TO_INPUT = re.compile(
    r"\b(the|these|provided) (provided |segment |yearly )?summar(y|ies)\b"
    r"|\b(this|the) (text|segment)s? (does|do) not\b"
    r"|contained within this (text|segment)|not (described|mentioned|provided|contained) in (the|this|these) (text|segment|summar)",
    re.I,
)
summaries["refers_to_input"] = summaries["summary"].map(lambda text: REFERS_TO_INPUT.search(text) is not None)
summaries["n_paragraphs"] = summaries["summary"].map(lambda text: len([p for p in re.split(r"\n\s*\n", text.strip()) if p.strip()]))
by_filings = summaries.groupby("n_filings").agg(firms=("gvkey", "size"), words_median=("n_words", "median"), words_max=("n_words", "max"),
                                                 bge_median=("n_bge_tokens", "median"), refers_to_input=("refers_to_input", "sum"))
display(by_filings)
print(f"Summaries referring to their own inputs: {int(summaries['refers_to_input'].sum()):,} of {len(summaries):,} ({summaries['refers_to_input'].mean():.1%})")
print(f"Summaries with more than one paragraph: {int(summaries['n_paragraphs'].gt(1).sum()):,}")


,firms,words_median,words_max,bge_median,refers_to_input
n_filings,,,,,
1,32,208.0,289,276.5,4
2,38,262.5,385,348.5,1
3,19,262.0,353,342.0,1
4,18,263.5,340,358.0,1
5,23,252.0,401,343.0,1
6,35,258.0,376,336.0,1
7,35,249.0,348,332.0,1
8,27,264.0,336,345.0,0
9,37,255.0,356,326.0,0


Summaries referring to their own inputs: 15 of 1,058 (1.4%)
Summaries with more than one paragraph: 0


## Step 7. Show examples

One firm with one, two, and three filings in the set, plus the shortest and the longest summary.


In [7]:
rng = np.random.default_rng(RANDOM_SEED)
picks = []
for n in sorted(summaries["n_filings"].unique()):
    group = summaries.loc[summaries["n_filings"].eq(n)]
    picks.append(group.iloc[[rng.integers(len(group))]])
picks += [summaries.nsmallest(1, "n_words"), summaries.nlargest(1, "n_words")]
examples = pd.concat(picks).drop_duplicates("gvkey")
examples = examples[["gvkey", "conm", "n_filings", "years", "n_words", "n_bge_tokens", "reduce_rounds", "refers_to_input", "summary"]]
examples.to_csv(OUTPUT_FOLDER / f"example_firm_summaries_{SET}_{FAMILY}_{VERSION}.csv", index=False)
for example in examples.itertuples(index=False):
    print(f"=== {example.conm} | {example.n_filings} filings {list(example.years)} | {example.n_words} words | {example.n_bge_tokens} bge tokens ===\n{example.summary}\n")


=== UNION PACIFIC RESOURCES GRP | 1 filings [1999] | 215 words | 276 bge tokens ===
Union Pacific Resources Group Inc. is primarily engaged in the exploration, development, and production of natural gas, natural gas liquids, and crude oil across the United States, Canada, Guatemala, and Venezuela, while also operating in the hard minerals sector through non-operated joint ventures and royalty arrangements for coal, industrial minerals, and trona mines in Wyoming. The company's core operating areas include U.S. Onshore, U.S. Offshore, Canada, and Other International. It sells its natural gas and crude oil production to customers, with a significant portion of domestic sales directed to Duke Energy Field Services, Inc. under a long-term agreement, while generating additional revenue from mineral rights and royalty arrangements associated with its coal and industrial mineral operations. The Minerals segment exploits fee mineral interests in coal, trona, and industrial minerals, producing 

## Step 8. Save the table and the check record

Rows of other runs already in the table are kept; rows of this set, family, and version are replaced by this run's.


In [8]:
table_columns = ["gvkey", "conm", "set", "family", "prompt_version", "n_filings", "years", "summary",
                 "n_words", "n_bge_tokens", "reduce_rounds", "trimmed", "refers_to_input", "run_started"]
new_rows = summaries[table_columns].copy()
new_rows["years"] = new_rows["years"].map(lambda years: [int(y) for y in years])
if TABLE_FILE.exists():
    existing = pd.read_parquet(TABLE_FILE)
    existing = existing.loc[~(existing["set"].eq(SET) & existing["family"].eq(FAMILY) & existing["prompt_version"].eq(VERSION))]
    table = pd.concat([existing, new_rows], ignore_index=True)
else:
    table = new_rows
assert not table.duplicated(["gvkey", "set", "family", "prompt_version"]).any()
table.to_parquet(TABLE_FILE, index=False)
check = pd.DataFrame([{
    "set": SET, "family": FAMILY, "prompt_version": VERSION, "n_firms": len(summaries), "n_filings_total": int(summaries["n_filings"].sum()),
    "words_median": float(summaries["n_words"].median()), "words_max": int(summaries["n_words"].max()), "bge_tokens_max": int(summaries["n_bge_tokens"].max()),
    "re_asked": int(summaries["reduce_rounds"].gt(1).sum()), "refers_to_input_share": float(summaries["refers_to_input"].mean()),
    "model_calls": record["model_calls"], "gpu": record["gpu"], "seconds": record["seconds"], "run_started": record["run_started"],
}])
check.to_csv(OUTPUT_FOLDER / f"firm_summary_check_{SET}_{FAMILY}_{VERSION}.csv", index=False)
print(f"Saved: {TABLE_FILE} ({len(table):,} rows; this run {len(new_rows):,})")
print(f"Saved: {OUTPUT_FOLDER / f'firm_summary_check_{SET}_{FAMILY}_{VERSION}.csv'}")
print(table.groupby(["set", "family", "prompt_version"]).size().rename("rows").to_string())


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Processed/firm_summaries.parquet (1,899 rows; this run 1,058)
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.5/firm_summary_check_all_years_summary_lines_of_business_item1-mapreduce-1.3.csv
set          family                     prompt_version     
all_years    summary_lines_of_business  item1-mapreduce-1.3    1058
three_years  summary_lines_of_business  item1-mapreduce-1.3     841
